In [1]:
pip install librosa soundfile numpy matplotlib tensorflow


     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ------ --------------------------------- 10.2/60.8 kB ? eta -:--:--
     ---------------------------------------- 60.8/60.8 kB 1.1 MB/s eta 0:00:00
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/167.0 kB ? eta -:--:--
     ------------------------- ------------ 112.6/167.0 kB 6.4 MB/s eta 0:00:01
     -------------------------------------- 167.0/167.0 kB 3.3 MB/s eta 0:00:00
  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [97]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense, Bidirectional, BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Set the paths for dataset and model
data_path = r"C:\Users\hp\Downloads\SSR Project\ravdess_dataset\audio_speech_actors_01-24"
model_filename = 'emotion.h5' 


In [98]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

emotions = {'01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
            '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'}

x_data, y_labels = [], []

data_path = r"C:\Users\hp\Downloads\SSR Project\ravdess_dataset\audio_speech_actors_01-24" 

if not os.path.exists(data_path):
    raise ValueError(f"Error: The path {data_path} does not exist.")
else:
    print("Contents of the data_path:", os.listdir(data_path))

def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc = np.mean(mfcc.T, axis=0)  
        return mfcc
    except Exception as e:
        print(f"Error extracting features from {file_path}: {e}")
        return np.array([]) 
for actor_folder in os.listdir(data_path):
    actor_path = os.path.join(data_path, actor_folder)
    if os.path.isdir(actor_path):  
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):  
                try:
                   
                    emotion_code = file.split("-")[2]  
                    emotion_label = emotions.get(emotion_code, 'unknown')  


                    feature = extract_features(os.path.join(actor_path, file))


                    if feature.shape[0] > 0:
                        x_data.append(feature)
                        y_labels.append(emotion_label)

                    print(f"Processed {file} in {actor_folder}: Feature shape {feature.shape}")

                except Exception as e:
                    print(f"Error processing {file} in {actor_folder}: {e}")


x_data = np.array(x_data)
y_labels = np.array(y_labels)


label_encoder = LabelEncoder()
y_labels = label_encoder.fit_transform(y_labels)


if len(x_data) > 0 and len(y_labels) > 0:
    #
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_labels, test_size=0.2, random_state=42)
    print(f"Training data shape: {x_train.shape}")
    print(f"Test data shape: {x_test.shape}")
else:
    print("Error: No data available to split. Check data path and feature extraction.")


Contents of the data_path: ['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05', 'Actor_06', 'Actor_07', 'Actor_08', 'Actor_09', 'Actor_10', 'Actor_11', 'Actor_12', 'Actor_13', 'Actor_14', 'Actor_15', 'Actor_16', 'Actor_17', 'Actor_18', 'Actor_19', 'Actor_20', 'Actor_21', 'Actor_22', 'Actor_23', 'Actor_24']
Processed 03-01-01-01-01-01-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-01-01-01-02-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-01-01-02-01-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-01-01-02-02-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-01-01-01-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-01-01-02-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-01-02-01-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-01-02-02-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-02-01-01-01.wav in Actor_01: Feature shape (13,)
Processed 03-01-02-02-01-02-01.wav in Actor_01: Feature shape (13,)
Proc

In [99]:
x_train = np.expand_dims(x_train, axis=2)
x_test = np.expand_dims(x_test, axis=2)

print(f"Reshaped training data shape: {x_train.shape}")
print(f"Reshaped testing data shape: {x_test.shape}")


Reshaped training data shape: (1152, 13, 1)
Reshaped testing data shape: (288, 13, 1)


In [100]:
model = Sequential()
model.add(LSTM(128, input_shape=(x_train.shape[1], 1), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(len(np.unique(y_labels)), activation='softmax'))  # Output layer with softmax

model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


c:\Programming\SSR\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_25 (LSTM)                  │ (None, 13, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_50 (Dropout)            │ (None, 13, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_26 (LSTM)                  │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_51 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_52 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 8)              │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,312 (462.16 KB)

 Trainable params: 118,312 (462.16 KB)

 Non-trainable params: 0 (0.00 B)

In [101]:
history = model.fit(x_train, y_train, epochs=200, batch_size=32, validation_data=(x_test, y_test))


Epoch 1/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.1660 - loss: 2.0443 - val_accuracy: 0.2014 - val_loss: 2.0012
Epoch 2/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1898 - loss: 2.0107 - val_accuracy: 0.2118 - val_loss: 1.9468
Epoch 3/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2353 - loss: 1.9333 - val_accuracy: 0.2604 - val_loss: 1.9103
Epoch 4/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2490 - loss: 1.8950 - val_accuracy: 0.2118 - val_loss: 1.9222
Epoch 5/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2258 - loss: 1.9321 - val_accuracy: 0.2118 - val_loss: 1.8974
Epoch 6/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2564 - loss: 1.8886 - val_accuracy: 0.2465 - val_loss: 1.8507
Epoch 7/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.2793 - loss: 1.8178 - val_accuracy: 0.2743 - val_loss: 1.8867
Epoch 8/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2790 - loss: 1.8525 - val_accuracy: 0.

In [102]:
model.save('emotion_model.h5')
print("Model saved to emotion_model.h5")


Model saved to emotion_model.h5


In [103]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)

print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy* 100:.2f}%")


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4698 - loss: 3.6336
Test Loss: 3.5917587280273438
Test Accuracy: 50.35%


In [105]:
from IPython.display import Audio
audio_file = r"C:\Users\hp\Downloads\SSR Project\ravdess_dataset\audio_speech_actors_01-24\Actor_01\03-01-08-02-01-02-01.wav"  
Audio(audio_file)

In [106]:
def predict_emotion(model, audio_file):
    feature = extract_features(audio_file)
    if feature.shape[0] > 0:
        feature = np.expand_dims(feature, axis=0)
        feature = np.expand_dims(feature, axis=2)

        
        prediction = model.predict(feature)
        emotion_index = np.argmax(prediction)
        emotion = label_encoder.inverse_transform([emotion_index])

        return emotion[0]
    else:
        return "Error: Unable to extract features"

predicted_emotion = predict_emotion(model, audio_file)
print(f"Predicted Emotion: {predicted_emotion}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step
Predicted Emotion: surprised
